# Title: *Airbnb Recommendation  System for Manchester Neighbourhoods Using K-Means Clustering and Foursquare API with Python*
### Developed by: *Olumide Olumayegun*

## Table of contents
1. Introduction and Busines Problem
2. Data
3. Methodology 
4. Results and Discussion
5. Conclusions

## 1. Introduction and Business Problem

### 1.1 Introduction

Manchester is a city and metropolitan borough in Greater Manchester, England, with a population of 547,627 as of 2018 (making it the fifth most populous English district) [1]. Manchester is the third most visited city in the UK, after London and Edinburgh. In 2019, it surpassed Edinburgh to become the second most visited city in the UK after London [2].

In this project we will try to find the best Airbnb listings in Manchester. Specifically, the system is target to clients interested in visiting Manchester and looking to book Airbnb accomodation in suitable neighbourhoods.

Since there are lots of Airbnb in Manchester, we will try to find the ones that meet the requirement of our clients in terms of the Airbnb features and proximity to some categories of venues in the neighbourhood.

We will deploy our data science skills to generate a list of promising Airbnb listings based on the clients's criteria. The generated listings can then be scrutinized further to filter out unwanted ones.


### 1.2 Business Problem

To build segregation and clustering models that will recommend to an individual suitable Airbnb accommodations in Manchester area based on available data of Airbnb listings and venues categories in Manchester neighbourhoods.

## 2. Data

The development and analysis of this system will require a variety of data sources. Below is a brief description of each dataset as well as links to the original sources where the data was downloaded from:

* **Airbnb listing for Greater Manchester, England:** This dataset contains information for Airbnb properties for the area of Greater Manchester. It is provided by *Inside AirBnb*. It was downloaded from the website at http://insideairbnb.com/get-the-data.html. It is available as *listings.csv* file under Greater Manchester, England, United Kingdom on the website. This file was downloaded and saved in the working directory of this project. The dataset contains information and metrics for Airbnb listings in Greater Manchester. Some useful observations in the data include: Listing id, neighbourhood group, neighbourhood, latitude and latitude of the listing, room type, price, number of reviews for the listing, and availability. K-Means algorithm will be applied on this dataset to cluster the Airbnb listings into different groups.
* **Geographical coordinates of Manchester neighbourhoods:** This is the latitude and longitude of the 32 neighbourhoods in Manchester area. This was obtained from ArcGIS using geocoder function in Python. A table of Manchester neighbourhoods and their latitude and longitude was created by pulling the neighbourhood geographical coordinates with Python calls. The neighbourhood geographical coordinates will be used to generate maps of Manchester with Folium. It will also form part of API request for venues in the neighbourhoods.
* **List and categories of venues in Manchester neighbourhoods:** This dataset contains important information about venues in each Manchester neighbourhood. This was obtained through API requests to Foursquare location data.  K-means algorithm will be applied on this dataset to cluster the Manchester neighbourhoods based on the categories of venues in the neighbourhood.

## 3. Methodology

### Download and import required libraries

In [82]:
import numpy as np

import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

import json #Library to handle JSON files

import requests # library to handle requests

# Use either geopy or geocoder

# for geopy:
#!conda install -c conda-forge geopy --yes # uncomment this line if you haven't installed geopy yet
#from geopy.geocoders import Nominatim # convert an address into latitude and longitude values

# We use geocoder in this notebook:
#!conda install -c conda-forge geocoder # uncomment this line if you haven't installed geocoder yet
import geocoder

from pandas import json_normalize # tranform JSON file into a pandas dataframe

# Matplotlib and associated plotting modules
import matplotlib.cm as cm
import matplotlib.colors as colors

# import k-means from clustering stage
from sklearn.cluster import KMeans

#!conda install -c conda-forge folium=0.5.0 --yes # uncomment this line if you dont have folium installed yet
import folium # map rendering library

print('Libraries imported.')

Libraries imported.


### Load and clean datasets

In [83]:
# Loading airbnb listings data for greater manchester compiled on 16 May, 2020
listings = pd.read_csv('../data/raw/manchester_listings.csv')
listings.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,68951,Whole House for rent,341682,Paul & Della,Bolton,Bolton District,53.56214,-2.53305,Entire home/apt,65,2,82,2019-10-03,0.82,1,196
1,85109,"Best Location! !, WIFI, En-suite, Balcony!",465854,Karyo,Manchester,City Centre,53.48074,-2.23186,Private room,50,2,434,2020-03-16,3.94,2,349
2,159189,1 room can be shared by two people,763769,Irfan,Rochdale,Rochdale District,53.61537,-2.19885,Shared room,55,1,0,NaN,NaN,1,349
3,283495,En-suite room in detached house,1476718,Alison,Rochdale,Rochdale District,53.56259,-2.21945,Private room,60,3,10,2018-08-05,0.16,1,300
4,299194,Cosy Garden Chalet for all seasons,1542010,Minh,Stockport,Stockport District,53.37600,-2.04462,Entire home/apt,50,2,255,2020-04-02,2.58,1,349


In [84]:
# Quick ckeck of the dataframe and attributes
listings.info()

<class 'pandas.DataFrame'>
RangeIndex: 4924 entries, 0 to 4923
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              4924 non-null   int64  
 1   name                            4923 non-null   str    
 2   host_id                         4924 non-null   int64  
 3   host_name                       4924 non-null   str    
 4   neighbourhood_group             4924 non-null   str    
 5   neighbourhood                   4924 non-null   str    
 6   latitude                        4924 non-null   float64
 7   longitude                       4924 non-null   float64
 8   room_type                       4924 non-null   str    
 9   price                           4924 non-null   int64  
 10  minimum_nights                  4924 non-null   int64  
 11  number_of_reviews               4924 non-null   int64  
 12  last_review                     3896 non-null

In [85]:
# Convert price to float data type
listings['price'] =listings['price'].astype(float)

# Check data types
listings.dtypes

id                                  int64
name                                  str
host_id                             int64
host_name                             str
neighbourhood_group                   str
neighbourhood                         str
latitude                          float64
longitude                         float64
room_type                             str
price                             float64
minimum_nights                      int64
number_of_reviews                   int64
last_review                           str
reviews_per_month                 float64
calculated_host_listings_count      int64
availability_365                    int64
dtype: object

In [86]:
# Removing unwanted/irrelevant observations
gr_man_listings = listings.drop(['name','host_id','host_name','last_review','reviews_per_month','calculated_host_listings_count'], axis=1)
gr_man_listings.head()

,id,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,availability_365
0,68951,Bolton,Bolton District,53.56214,-2.53305,Entire home/apt,65.0,2,82,196
1,85109,Manchester,City Centre,53.48074,-2.23186,Private room,50.0,2,434,349
2,159189,Rochdale,Rochdale District,53.61537,-2.19885,Shared room,55.0,1,0,349
3,283495,Rochdale,Rochdale District,53.56259,-2.21945,Private room,60.0,3,10,300
4,299194,Stockport,Stockport District,53.37600,-2.04462,Entire home/apt,50.0,2,255,349


In [87]:
gr_man_listings.shape

(4924, 10)

In [88]:
# Check that all listings are unique
len(gr_man_listings['id'].unique().tolist()) == gr_man_listings.shape[0]

True

No duplicate listings

In [89]:
# Check for missing values
gr_man_listings.isnull().sum()

id                     0
neighbourhood_group    0
neighbourhood          0
latitude               0
longitude              0
room_type              0
price                  0
minimum_nights         0
number_of_reviews      0
availability_365       0
dtype: int64

No missing value in any of the column

### Descriptive statistical analysis

In [90]:
# Descriptive statistical analysis
gr_man_listings.describe(include='all')

,id,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,availability_365
count,4.924000e+03,4924,4924,4924.000000,4924.000000,4924,4924.000000,4924.000000,4924.000000,4924.000000
unique,NaN,10,41,NaN,NaN,4,NaN,NaN,NaN,NaN
top,NaN,Manchester,Salford District,NaN,NaN,Private room,NaN,NaN,NaN,NaN
freq,NaN,2823,787,NaN,NaN,2412,NaN,NaN,NaN,NaN
mean,2.855575e+07,NaN,NaN,53.472945,-2.249493,NaN,88.091998,3.518278,27.871649,153.741470
std,1.143300e+07,NaN,NaN,0.045497,0.081715,NaN,173.412694,15.870997,51.629740,138.836712
min,6.895100e+04,NaN,NaN,53.345830,-2.708880,NaN,0.000000,1.000000,0.000000,0.000000
25%,1.996664e+07,NaN,NaN,53.448817,-2.275375,NaN,30.000000,1.000000,1.000000,0.750000
50%,3.044096e+07,NaN,NaN,53.474685,-2.239015,NaN,52.000000,1.000000,7.000000,124.500000
75%,3.888886e+07,NaN,NaN,53.488280,-2.218380,NaN,90.000000,2.000000,30.000000,323.000000


The data showed that there are 4924 Airbnb listings in Greater Manchester as at 16th of May 2020. 
There are 41 neighbourhoods dstributed across 10 neighbourhood groups (or Borough). Salford District neighbourhood has the highest number of Airbnb listings of 787. More than half of the listings are in the Machester neighbourhood group. 
The rooms are grouped into 4 types with the most popular room type being private room.

In [91]:
# Room types
gr_man_listings['room_type'].value_counts()

room_type
Private room       2412
Entire home/apt    2385
Hotel room           64
Shared room          63
Name: count, dtype: int64

### Visualisation of the Airbnb listings on the map of Greater Manchester

Get the latitude and longitude of Greater Manchester

In [97]:
address1 = 'Greater Manchester, England'
geolocator1 = geocoder.arcgis(address1)
latitude_gr_man = geolocator1.latlng[0]
longitude_gr_man = geolocator1.latlng[1]
print('The geograpical coordinate of Greater Manchester are {} lat, {} long.'.format(latitude_gr_man, longitude_gr_man))

The geograpical coordinate of Greater Manchester are 53.450000000000045 lat, -2.2333299999999667 long.


In [98]:
# create map of Greater Manchester using latitude and longitude values
map_gr_man = folium.Map(location=[latitude_gr_man, longitude_gr_man], zoom_start=10)

# add markers to map
for lat, lng, borough, neighborhood in zip(gr_man_listings['latitude'], gr_man_listings['longitude'], gr_man_listings['neighbourhood_group'], gr_man_listings['neighbourhood']):
    label = '{}, {}'.format(neighborhood, borough)
    label = folium.Popup(label, parse_html=True)
    folium.CircleMarker(
        [lat, lng],
        radius=1,
        popup=label,
        color='blue',
        fill=True,
        fill_color='#3186cc',
        fill_opacity=0.7,
        parse_html=False).add_to(map_gr_man)  
    
map_gr_man.save('../maps/manchester_listings.png')
#map_gr_man

[Open Interactive Map](../maps/manchester_listings.png)

There is dense concentration of listings around the Manchester City area.
Let's check the listings according to the neighbourhood groups

In [99]:
df_count = gr_man_listings.groupby('neighbourhood_group')['id'].count()
df_count

neighbourhood_group
Bolton         133
Bury           130
Manchester    2823
Oldham         137
Rochdale        52
Salford        787
Stockport      259
Tameside       111
Trafford       407
Wigan           85
Name: id, dtype: int64

Manchester city is a popular city with visitors such as footbool fans, business travels, tourists, e.t.c.

So let assume that we are only intrested in Airbnb listed in Manchester City neighbourhoods.

In [100]:
man_listings = gr_man_listings[gr_man_listings['neighbourhood_group']=='Manchester'].reset_index(drop=True)
man_listings.head()

,id,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,availability_365
0,85109,Manchester,City Centre,53.48074,-2.23186,Private room,50.0,2,434,349
1,310742,Manchester,Ancoats and Clayton,53.48251,-2.22802,Private room,37.0,180,64,0
2,332580,Manchester,City Centre,53.47859,-2.23195,Private room,48.0,2,322,0
3,360142,Manchester,Moss Side,53.45678,-2.23513,Private room,32.0,3,47,349
4,363327,Manchester,Moss Side,53.45326,-2.23971,Private room,20.0,1,27,75


In [101]:
man_listings.drop(['neighbourhood_group'], axis=1, inplace=True)
man_listings.head()

,id,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,availability_365
0,85109,City Centre,53.48074,-2.23186,Private room,50.0,2,434,349
1,310742,Ancoats and Clayton,53.48251,-2.22802,Private room,37.0,180,64,0
2,332580,City Centre,53.47859,-2.23195,Private room,48.0,2,322,0
3,360142,Moss Side,53.45678,-2.23513,Private room,32.0,3,47,349
4,363327,Moss Side,53.45326,-2.23971,Private room,20.0,1,27,75


### K-means segregation and clustering of Airbnb in Manchester

Let's segregate the listings in Manchester into ten clusters based on the room type, price, minimum nights, number of reviews and availability using KMeans clustering algorithm.

The groupings should not be influenced by the geographical location of the listings. Hence, we won't be using the geographical coordinates as features for the clustering algorithm. The neighbourhood location of the listings will only be taken into consideration through the available amenities/venues in the neighbourhood. This will be investigated later through neighbourhood segregation and clustering.


In [39]:
# Convert categorical variable into one hot encoding
room_type_onehot = pd.get_dummies(man_listings[['room_type']], prefix="", prefix_sep="")
room_type_onehot.drop(['Entire home/apt'], axis=1, inplace=True)
room_type_onehot.head()

,Hotel room,Private room,Shared room
0,False,True,False
1,False,True,False
2,False,True,False
3,False,True,False
4,False,True,False


In [40]:
man_data = pd.concat([room_type_onehot, man_listings['price'], man_listings['minimum_nights'], man_listings['number_of_reviews'], man_listings['availability_365']], axis=1)
man_data.head()

,Hotel room,Private room,Shared room,price,minimum_nights,number_of_reviews,availability_365
0,False,True,False,50.0,2,434,349
1,False,True,False,37.0,180,64,0
2,False,True,False,48.0,2,322,0
3,False,True,False,32.0,3,47,349
4,False,True,False,20.0,1,27,75


Normalise the data over the standard deviation

In [41]:
from sklearn.preprocessing import StandardScaler
X_man = man_data.values[:]
listings_scaler = StandardScaler()
X_man = listings_scaler.fit_transform(X_man)
X_man

array([[-0.10187926,  1.05125897, -0.12582938, ..., -0.11632769,
         7.07753144,  1.4602455 ],
       [-0.10187926,  1.05125897, -0.12582938, ..., 10.24601369,
         0.58001233, -1.0571225 ],
       [-0.10187926,  1.05125897, -0.12582938, ..., -0.11632769,
         5.11071485, -1.0571225 ],
       ...,
       [-0.10187926, -0.9512404 , -0.12582938, ..., -0.17454309,
        -0.54388286,  1.4602455 ],
       [-0.10187926, -0.9512404 , -0.12582938, ..., -0.17454309,
        -0.54388286,  1.4602455 ],
       [-0.10187926, -0.9512404 , -0.12582938, ...,  0.17474931,
        -0.54388286,  0.12582407]], shape=(2823, 7))

Modelling - Let's apply k-means on the dataset and group the listings into 10 clusters

In [42]:
clusterNum = 10
k_means_man = KMeans(init = "k-means++", n_clusters = clusterNum, n_init = 12, random_state=7)
k_means_man.fit(X_man)
labels = k_means_man.labels_
print(labels)

[7 0 7 ... 3 3 1]


In [43]:
# Asign the labels to each row of man_listings dataframe
man_listings['bnb_cluster'] = labels
man_listings.head()

,id,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,availability_365,bnb_cluster
0,85109,City Centre,53.48074,-2.23186,Private room,50.0,2,434,349,7
1,310742,Ancoats and Clayton,53.48251,-2.22802,Private room,37.0,180,64,0,0
2,332580,City Centre,53.47859,-2.23195,Private room,48.0,2,322,0,7
3,360142,Moss Side,53.45678,-2.23513,Private room,32.0,3,47,349,9
4,363327,Moss Side,53.45326,-2.23971,Private room,20.0,1,27,75,2


In [44]:
man_listings.groupby('bnb_cluster')['id'].count()

bnb_cluster
0     18
1    856
2    850
3    433
4     44
5     29
6     52
7    142
8      4
9    395
Name: id, dtype: int64

Visualising the resulting clustering of the Airbnb listings

In [45]:
# Get the latitude and longitude of Manchester City
address2 = 'Manchester City, England'
geolocator2 = geocoder.arcgis(address2)
latitude_man = geolocator2.latlng[0]
longitude_man = geolocator2.latlng[1]
print('The geograpical coordinate of Manchester City are {} lat, {} long.'.format(latitude_man, longitude_man))

The geograpical coordinate of Manchester City are 53.47895624300003 lat, -2.2452757659999634 long.


In [46]:
# Visualizing the resulting clusters
# create map
map_man_clusters = folium.Map(location=[latitude_man, longitude_man], zoom_start=11)

# set color scheme for the clusters
x = np.arange(clusterNum)
ys = [i + x + (i*x)**2 for i in range(clusterNum)]
colors_array = cm.rainbow(np.linspace(0, 1, len(ys)))
rainbow = [colors.rgb2hex(i) for i in colors_array]

# add markers to the map
markers_colors = []
for lat, lon, poi, cluster in zip(man_listings['latitude'], man_listings['longitude'], man_listings['neighbourhood'], man_listings['bnb_cluster']):
    label = folium.Popup(str(poi) + ' Cluster ' + str(cluster), parse_html=True)
    folium.CircleMarker(
        [lat, lon],
        radius=1,
        popup=label,
        color=rainbow[cluster-1],
        fill=True,
        fill_color=rainbow[cluster-1],
        fill_opacity=0.7).add_to(map_man_clusters)
       
map_man_clusters

Therefore, based on a customers preferences for Airbnb, we can suggest a group of bnb to the   customer by using the trained KMeans model to predict appropriate cluster.

Suppose the customer is also interested in the facilities/venues in the neighbourhood of the listings in order to narrow down the choices since there are still many options to choose from within each clusters.

We will approach this by also segregating and clustering the Manchester City neighbourhoods based on the venues.

### K-Means segregation and clustering of neighbourhoods in Manchester

In [47]:
# Create a dataframe of Manchester City neighbourhood from the Airbnb listings
neigh = pd.DataFrame(man_listings['neighbourhood'].unique(), columns=['Neighbourhood'])
neigh

,Neighbourhood
0,City Centre
1,Ancoats and Clayton
2,Moss Side
3,Bradford
4,Hulme
5,Chorlton
6,Didsbury West
7,Old Moat
8,Burnage
9,Levenshulme


There are 32 neighbourhoods in Manchester City.

Let's obtain the latitude and longitude of each neighbourhood and add to the dataframe

In [48]:
latitude = np.zeros((neigh.shape[0])) # Initialise latitude values with zeros
longitude = np.zeros((neigh.shape[0])) # Initialise longitude values with zeros

for i, nei in enumerate(neigh["Neighbourhood"]):
    lat_lng_coords = None # Initialialise variable to none
    
    while (lat_lng_coords is None): #Loop until you get the coordinates
        g = geocoder.arcgis('{}, Manchester, England'.format(nei)) # Note: goecoder.google does not work, use geocoder.arcgis instead
        lat_lng_coords = g.latlng
            
    latitude[i] = lat_lng_coords[0]
    longitude[i] = lat_lng_coords[1]

neigh["Latitude"] = latitude    # Add latitude column to the dataframe
neigh["Longitude"] = longitude  # Add longitude column to the dataframe
neigh

,Neighbourhood,Latitude,Longitude
0,City Centre,53.480365,-2.241984
1,Ancoats and Clayton,53.484372,-2.182543
2,Moss Side,53.459014,-2.244017
3,Bradford,53.488989,-2.206103
4,Hulme,53.467052,-2.246985
5,Chorlton,53.435050,-2.263100
6,Didsbury West,53.423368,-2.246432
7,Old Moat,53.435577,-2.235874
8,Burnage,53.426227,-2.203678
9,Levenshulme,53.441961,-2.185025


In [49]:
# create map of Manchester neighbourhoods using latitude and longitude values
map_neigh = folium.Map(location=[latitude_man, longitude_man], zoom_start=11)

# add markers to map
for lat, lng, label in zip(neigh['Latitude'], neigh['Longitude'], neigh['Neighbourhood']):
    label = folium.Popup(label, parse_html=True)
    folium.CircleMarker(
        [lat, lng],
        radius=5,
        popup=label,
        color='blue',
        fill=True,
        fill_color='#3186cc',
        fill_opacity=0.7,
        parse_html=False).add_to(map_neigh)  
    
map_neigh

Foursquare API credential and version

In [52]:
CLIENT_ID = 'LRICOQAGACQJCGVLBPKE2ZAQ1UYMWENOKBWLTMFT2UDWRCLK' # your Foursquare ID
CLIENT_SECRET = 'A3B55ZKMIABIR1D3LIPQL4TG2AIHHHVZPI5KY0WDR0B5YDEM' # your Foursquare Secret
VERSION = '20250301' # Foursquare API version
#SERVICE_API = 'JVIA3CYV0CJ0NUMBAD3IWS1MBUYHL0YNU5GF5QT1JYRK2YOZ'

print('Your credentails:')
#print('CLIENT_ID: ' + CLIENT_ID)
#print('CLIENT_SECRET:' + CLIENT_SECRET)

LIMIT = 100 # limit of number of venues returned by Foursquare API

radius = 500 # define radius

Your credentails:


In [ ]:
# function that extracts the category of the venue
#def get_category_type(row):
    #try:
        #categories_list = row['categories']
    #except:
        #categories_list = row['venue.categories']
        
    #if len(categories_list) == 0:
        #return None
    #else:
        #return categories_list[0]['name']

Create a function that get the venues in each neighbourhood

In [53]:
def getNearbyVenues(names, latitudes, longitudes, radius=500):
    
    venues_list=[]
    for name, lat, lng in zip(names, latitudes, longitudes):
        print(name)
            
        # create the API request URL
        url = 'https://api.foursquare.com/v2/venues/explore?&client_id={}&client_secret={}&v={}&ll={},{}&radius={}&limit={}'.format(
            CLIENT_ID, 
            CLIENT_SECRET, 
            VERSION, 
            lat, 
            lng, 
            radius, 
            LIMIT)
            
        # make the GET request
        results = requests.get(url).json()["response"]['groups'][0]['items']
        
        # return only relevant information for each nearby venue
        venues_list.append([(
            name, 
            lat, 
            lng, 
            v['venue']['name'], 
            v['venue']['location']['lat'], 
            v['venue']['location']['lng'],  
            v['venue']['categories'][0]['name']) for v in results])

    nearby_venues = pd.DataFrame([item for venue_list in venues_list for item in venue_list])
    nearby_venues.columns = ['Neighborhood', 
                  'Neighborhood Latitude', 
                  'Neighborhood Longitude', 
                  'Venue', 
                  'Venue Latitude', 
                  'Venue Longitude', 
                  'Venue Category']
    
    return(nearby_venues)

In [54]:
manchester_venues = getNearbyVenues(names=neigh['Neighbourhood'],
                                   latitudes=neigh['Latitude'],
                                   longitudes=neigh['Longitude']
                                  )

City Centre
Ancoats and Clayton
Moss Side
Bradford
Hulme
Chorlton
Didsbury West
Old Moat
Burnage
Levenshulme
Didsbury East
Withington
Chorlton Park
Longsight
Whalley Range
Moston
Cheetham
Higher Blackley
Rusholme
Ardwick
Brooklands
Gorton South
Gorton North
Fallowfield
Miles Platting and Newton Heath
Baguley
Northenden
Harpurhey
Crumpsall
Woodhouse Park
Sharston
Charlestown


In [55]:
# Let's check the size of the resulting dataframe
print(manchester_venues.shape)
manchester_venues.head(20)

(360, 7)


,Neighborhood,Neighborhood Latitude,Neighborhood Longitude,Venue,Venue Latitude,Venue Longitude,Venue Category
0,City Centre,53.480365,-2.241984,Café Beermoth,53.481359,-2.241902,Bar
1,City Centre,53.480365,-2.241984,Manchester Art Gallery,53.478882,-2.241817,Art Gallery
2,City Centre,53.480365,-2.241984,King Street Townhouse,53.480050,-2.243268,Hotel
3,City Centre,53.480365,-2.241984,Knoops,53.480869,-2.245054,Café
4,City Centre,53.480365,-2.241984,Pret a Manger,53.481873,-2.244276,Sandwich Spot
5,City Centre,53.480365,-2.241984,Hotel Gotham,53.480767,-2.242235,Hotel
6,City Centre,53.480365,-2.241984,Hampton and Voúis,53.479747,-2.243931,Café
7,City Centre,53.480365,-2.241984,Flight Club,53.480574,-2.242946,Sports Bar
8,City Centre,53.480365,-2.241984,Salut,53.479353,-2.242251,Wine Bar
9,City Centre,53.480365,-2.241984,Bold Street Coffee,53.480431,-2.244998,Coffee Shop


In [56]:
# Let's check how many venues were returned for each neighborhood
manchester_venues.groupby('Neighborhood').count()

,Neighborhood Latitude,Neighborhood Longitude,Venue,Venue Latitude,Venue Longitude,Venue Category
Neighborhood,,,,,,
Ancoats and Clayton,5,5,5,5,5,5
Ardwick,6,6,6,6,6,6
Baguley,14,14,14,14,14,14
Bradford,4,4,4,4,4,4
Brooklands,4,4,4,4,4,4
Burnage,3,3,3,3,3,3
Charlestown,1,1,1,1,1,1
Cheetham,1,1,1,1,1,1
Chorlton,6,6,6,6,6,6


In [57]:
# Let's find out how many unique categories can be curated from all the returned venues
print('There are {} uniques categories.'.format(len(manchester_venues['Venue Category'].unique())))

There are 119 uniques categories.


##### Analysing each neighbourhood

In [58]:
# one hot encoding
manchester_onehot = pd.get_dummies(manchester_venues[['Venue Category']], prefix="", prefix_sep="")

# add neighborhood column back to dataframe and insert as the first column
manchester_onehot.insert(loc=0, column='Neighbourhood', value=manchester_venues['Neighborhood'])

manchester_onehot.head()

,Neighbourhood,American Restaurant,Antique Store,Argentinian Restaurant,Art Gallery,Arts and Entertainment,Asian Restaurant,Australian Restaurant,Bakery,Bar,Beer Bar,Beer Store,Betting Shop,Bicycle Store,Bookstore,Bowling Alley,Brazilian Restaurant,Bridal Store,Bubble Tea Shop,Burger Joint,Burrito Restaurant,Bus Station,Bus Stop,Café,Canal,Caribbean Restaurant,Casino,Chinese Restaurant,Chocolate Store,Clothing Store,Cocktail Bar,Coffee Shop,Comic Book Store,Concert Hall,Convenience Store,Cosmetics Store,Cultural Center,Cupcake Shop,Deli,Department Store,Dessert Shop,Diner,Discount Store,Donut Shop,Electronics Store,English Restaurant,Escape Room,Ethiopian Restaurant,Event Space,Eyecare Store,Falafel Restaurant,Farm,Fast Food Restaurant,Fish and Chips Shop,Flea Market,Flower Store,Fuel Station,Furniture and Home Store,Garden,Garden Center,Gay Bar,Grocery Store,Hardware Store,Hotel,Ice Cream Parlor,Indian Restaurant,Irish Pub,Italian Restaurant,Japanese Restaurant,Jewelry Store,Lake,Latin American Restaurant,Liquor Store,Market,Middle Eastern Restaurant,Mobile Phone Store,Movie Theater,Museum,Music Venue,Pakistani Restaurant,Park,Pastry Shop,Pedestrian Plaza,Performing Arts Venue,Persian Restaurant,Pet Supplies Store,Pharmacy,Pie Shop,Pier,Pizzeria,Plaza,Portuguese Restaurant,Pub,Ramen Restaurant,Record Store,Restaurant,Retail,Sandwich Spot,Scenic Lookout,Shoe Store,Shopping Mall,Shopping Plaza,Speakeasy,Sporting Goods Retail,Sports Bar,Supermarket,Sushi Restaurant,Tanning Salon,Tapas Restaurant,Tea Room,Thai Restaurant,Theater,Toy Store,Tram Station,Turkish Restaurant,Vegan and Vegetarian Restaurant,Vietnamese Restaurant,Wine Bar,Wine Store,Women's Store
0,City Centre,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,City Centre,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,City Centre,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,City Centre,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,Fal

In [59]:
#Examine the new dataframe
manchester_onehot.shape

(360, 120)

In [60]:
# Let's group rows by neighborhood and by taking the mean of the frequency of occurrence of each category
manchester_grouped = manchester_onehot.groupby('Neighbourhood').mean().reset_index()
manchester_grouped

,Neighbourhood,American Restaurant,Antique Store,Argentinian Restaurant,Art Gallery,Arts and Entertainment,Asian Restaurant,Australian Restaurant,Bakery,Bar,Beer Bar,Beer Store,Betting Shop,Bicycle Store,Bookstore,Bowling Alley,Brazilian Restaurant,Bridal Store,Bubble Tea Shop,Burger Joint,Burrito Restaurant,Bus Station,Bus Stop,Café,Canal,Caribbean Restaurant,Casino,Chinese Restaurant,Chocolate Store,Clothing Store,Cocktail Bar,Coffee Shop,Comic Book Store,Concert Hall,Convenience Store,Cosmetics Store,Cultural Center,Cupcake Shop,Deli,Department Store,Dessert Shop,Diner,Discount Store,Donut Shop,Electronics Store,English Restaurant,Escape Room,Ethiopian Restaurant,Event Space,Eyecare Store,Falafel Restaurant,Farm,Fast Food Restaurant,Fish and Chips Shop,Flea Market,Flower Store,Fuel Station,Furniture and Home Store,Garden,Garden Center,Gay Bar,Grocery Store,Hardware Store,Hotel,Ice Cream Parlor,Indian Restaurant,Irish Pub,Italian Restaurant,Japanese Restaurant,Jewelry Store,Lake,Latin American Restaurant,Liquor Store,Market,Middle Eastern Restaurant,Mobile Phone Store,Movie Theater,Museum,Music Venue,Pakistani Restaurant,Park,Pastry Shop,Pedestrian Plaza,Performing Arts Venue,Persian Restaurant,Pet Supplies Store,Pharmacy,Pie Shop,Pier,Pizzeria,Plaza,Portuguese Restaurant,Pub,Ramen Restaurant,Record Store,Restaurant,Retail,Sandwich Spot,Scenic Lookout,Shoe Store,Shopping Mall,Shopping Plaza,Speakeasy,Sporting Goods Retail,Sports Bar,Supermarket,Sushi Restaurant,Tanning Salon,Tapas Restaurant,Tea Room,Thai Restaurant,Theater,Toy Store,Tram Station,Turkish Restaurant,Vegan and Vegetarian Restaurant,Vietnamese Restaurant,Wine Bar,Wine Store,Women's Store
0,Ancoats and Clayton,0.00000,0.000000,0.00,0.00,0.00000,0.000000,0.00,0.000000,0.000000,0.00,0.000000,0.000000,0.00,0.00,0.000000,0.00,0.00,0.00,0.000000,0.00,0.000000,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.00,0.000000,0.00000,0.000000,0.00,0.00,0.000000,0.2,0.0,0.000000,0.000000,0.0000,0.00000,0.000000,0.000000,0.000000,0.00,0.00,0.00000,0.00,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.00,0.200000,0.000000,0.000000,0.0,0.00000,0.000000,0.000,0.000000,0.2,0.000000,0.00000,0.000000,0.000000,0.200000,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.00,0.000000,0.00,0.000000,0.000000,0.00,0.00,0.000000,0.20,0.000000,0.0,0.00,0.000000,0.000000,0.00,0.00000,0.000000,0.000000,0.000000,0.000000,0.00,0.00,0.00,0.00,0.00,0.000000,0.000,0.000000,0.0,0.00,0.000000,0.00000
1,Ardwick,0.00000,0.000000,0.00,0.00,0.00000,0.000000,0.00,0.000000,0.000000,0.00,0.000000,0.000000,0.00,0.00,0.000000,0.00,0.00,0.00,0.000000,0.00,0.000000,0.000000,0.166667,0.00,0.000000,0.000000,0.000000,0.00,0.000000,0.00000,0.000000,0.00,0.00,0.000000,0.0,0.0,0.000000,0.166667,0.0000,0.00000,0.000000,0.000000,0.000000,0.00,0.00,0.00000,0.00,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.166667,0.000000,0.000000,0.000000,0.000000,0.166667,0.000000,0.000000,0.0,0.000000,0.00,0.000000,0.000000,0.000000,0.0,0.00000,0.000000,0.000,0.000000,0.0,0.000000,0.00000,0.166667,0.000000,0.166667,0.00,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,0.00,0.000000,0.00,0.000000,0.000000,0.00,0.00,0.000000,0.00,0.000000,0.0,0.00,0.000000,0.000000,0.00,0.00000,0.000000,0.000000,0.000000,0.000000,0.00,0.00,0.00,0.00,0.00,0.000000,0.000,0.000000,0.0,0.00,0.000000,0.00000
2,Baguley,0.00000,0.000000,0.00,0.00,0.00000,0.000000,0.00,0.000000,0.000000,0.00,0.000000,0.000000,0.00,0.00,0.000000,0.00,0.00,0.00,0.000000,0.00,0.000000,0.000000,0.000000,0.00,0.000000,0.000000,0.000000,0.00,0.071429,0.00000,0.071429,0.00,0.00,0.000000,0.0,0.0,0.000000,0.000000,0.0000,0.00000,0.000000,0.000000,0.071429,0.00,0.00,0.00000,0.00,0.00,0.000000,0.000000,0.000000,0.071429,0.000000,0.000000,0.000000,0.142857,0.071429,0.071429,0.000000,0.000000,0.000000,0.071429,0.000000,0.0,0.000000,0.00,0.000000,0.000

In [61]:
# Confirm the new size
manchester_grouped.shape

(32, 120)

In [62]:
# Let's print each neighborhood along with the top 5 most common venues
num_top_venues = 5

for hood in manchester_grouped['Neighbourhood']:
    print("----"+hood+"----")
    temp = manchester_grouped[manchester_grouped['Neighbourhood'] == hood].T.reset_index()
    temp.columns = ['venue','freq']
    temp = temp.iloc[1:]
    temp['freq'] = temp['freq'].astype(float)
    temp = temp.round({'freq': 2})
    print(temp.sort_values('freq', ascending=False).reset_index(drop=True).head(num_top_venues))
    print('\n')

----Ancoats and Clayton----
                venue  freq
0     Cosmetics Store   0.2
1  Mobile Phone Store   0.2
2  Italian Restaurant   0.2
3              Retail   0.2
4                Park   0.2


----Ardwick----
          venue  freq
0          Café  0.17
1          Deli  0.17
2  Fuel Station  0.17
3   Music Venue  0.17
4          Park  0.17


----Baguley----
                  venue  freq
0           Supermarket  0.21
1          Fuel Station  0.14
2  Fast Food Restaurant  0.07
3            Donut Shop  0.07
4                Garden  0.07


----Bradford----
             venue  freq
0            Canal  0.25
1     Tram Station  0.25
2      Music Venue  0.25
3  Thai Restaurant  0.25
4      Art Gallery  0.00


----Brooklands----
          venue  freq
0          Café  0.25
1  Bridal Store  0.25
2        Bakery  0.25
3      Pizzeria  0.25
4   Art Gallery  0.00


----Burnage----
                    venue  freq
0       Convenience Store  0.33
1             Supermarket  0.33
2            Liquor 

Let's put that into pandas dataframe

In [65]:
# First, let's write a function to sort the venues in descending order
def return_most_common_venues(row, num_top_venues):
    row_categories = row.iloc[1:]
    row_categories_sorted = row_categories.sort_values(ascending=False)
    
    return row_categories_sorted.index.values[0:num_top_venues]

In [66]:
# Now let's create the new dataframe and display the top 10 venues for each neighborhood.
num_top_venues = 10

indicators = ['st', 'nd', 'rd']

# create columns according to number of top venues
columns = ['Neighbourhood']
for ind in np.arange(num_top_venues):
    try:
        columns.append('{}{} Most Common Venue'.format(ind+1, indicators[ind]))
    except:
        columns.append('{}th Most Common Venue'.format(ind+1))

# create a new dataframe
neighborhoods_venues_sorted = pd.DataFrame(columns=columns)
neighborhoods_venues_sorted['Neighbourhood'] = manchester_grouped['Neighbourhood']

for ind in np.arange(manchester_grouped.shape[0]):
    neighborhoods_venues_sorted.iloc[ind, 1:] = return_most_common_venues(manchester_grouped.iloc[ind, :], num_top_venues)

neighborhoods_venues_sorted

,Neighbourhood,1st Most Common Venue,2nd Most Common Venue,3rd Most Common Venue,4th Most Common Venue,5th Most Common Venue,6th Most Common Venue,7th Most Common Venue,8th Most Common Venue,9th Most Common Venue,10th Most Common Venue
0,Ancoats and Clayton,Cosmetics Store,Italian Restaurant,Retail,Park,Mobile Phone Store,American Restaurant,Museum,Pie Shop,Pharmacy,Pet Supplies Store
1,Ardwick,Grocery Store,Fuel Station,Deli,Café,Park,Music Venue,Museum,Pier,Pie Shop,Pharmacy
2,Baguley,Supermarket,Fuel Station,Clothing Store,Hardware Store,Coffee Shop,Pet Supplies Store,Garden,Tram Station,Furniture and Home Store,Donut Shop
3,Bradford,Tram Station,Canal,Thai Restaurant,Music Venue,American Restaurant,Movie Theater,Pie Shop,Pharmacy,Pet Supplies Store,Persian Restaurant
4,Brooklands,Bridal Store,Pizzeria,Bakery,Café,Museum,Pier,Pie Shop,Pharmacy,Pet Supplies Store,Persian Restaurant
5,Burnage,Supermarket,Convenience Store,Liquor Store,Movie Theater,Pie Shop,Pharmacy,Pet Supplies Store,Persian Restaurant,Performing Arts Venue,Pedestrian Plaza
6,Charlestown,Coffee Shop,Grocery Store,Pizzeria,Pier,Pie Shop,Pharmacy,Pet Supplies Store,Persian Restaurant,Performing Arts Venue,Pedestrian Plaza
7,Cheetham,Café,American Restaurant,Plaza,Pier,Pie Shop,Pharmacy,Pet Supplies Store,Persian Restaurant,Performing Arts Venue,Pedestrian Plaza
8,Chorlton,Tram Station,Flower Store,Jewelry Store,Park,Music Venue,American Restaurant,Movie Theater,Pie Shop,Pharmacy,Pet Supplies Store
9,Chorlton Park,Grocery Store,Sports Bar,Bar,Park,Caribbean Restaurant,Falafel Restaurant,Fish and Chips Shop,Fuel Station,Pizzeria,Japanese Restaurant


#### Clustering neighbourhoods

 Runs k-means to cluster the neighbourhood into 5 clusters

In [70]:
# set number of clusters
kclusters = 5

manchester_grouped_clustering = manchester_grouped.drop('Neighbourhood', axis=1)

# run k-means clustering
neigh_kmeans = KMeans(n_clusters=kclusters, random_state=0).fit(manchester_grouped_clustering)

# check cluster labels generated for each row in the dataframe
neigh_kmeans.labels_[0:10] 

array([2, 2, 0, 2, 2, 0, 1, 3, 2, 2], dtype=int32)

Creating a new dataframe that includes the cluster as well as the top 10 venues for each neighborhood

In [71]:
# add clustering labels
neighborhoods_venues_sorted.insert(0, 'Cluster Labels', neigh_kmeans.labels_)

manchester_merged = neigh

# merge manchester_grouped with manchester_data to add latitude/longitude for each neighborhood
manchester_merged = manchester_merged.join(neighborhoods_venues_sorted.set_index('Neighbourhood'), on='Neighbourhood')

manchester_merged # check the last columns!


,Neighbourhood,Latitude,Longitude,Cluster Labels,1st Most Common Venue,2nd Most Common Venue,3rd Most Common Venue,4th Most Common Venue,5th Most Common Venue,6th Most Common Venue,7th Most Common Venue,8th Most Common Venue,9th Most Common Venue,10th Most Common Venue
0,City Centre,53.480365,-2.241984,2,Coffee Shop,Hotel,Bar,Café,Pub,Plaza,Asian Restaurant,Italian Restaurant,Clothing Store,Bookstore
1,Ancoats and Clayton,53.484372,-2.182543,2,Cosmetics Store,Italian Restaurant,Retail,Park,Mobile Phone Store,American Restaurant,Museum,Pie Shop,Pharmacy,Pet Supplies Store
2,Moss Side,53.459014,-2.244017,0,Discount Store,Supermarket,Fuel Station,Farm,Fast Food Restaurant,Pastry Shop,Music Venue,Pakistani Restaurant,Park,Pedestrian Plaza
3,Bradford,53.488989,-2.206103,2,Tram Station,Canal,Thai Restaurant,Music Venue,American Restaurant,Movie Theater,Pie Shop,Pharmacy,Pet Supplies Store,Persian Restaurant
4,Hulme,53.467052,-2.246985,2,Convenience Store,Café,Park,Tanning Salon,Pharmacy,Performing Arts Venue,Chinese Restaurant,Pedestrian Plaza,Caribbean Restaurant,Fast Food Restaurant
5,Chorlton,53.435050,-2.263100,2,Tram Station,Flower Store,Jewelry Store,Park,Music Venue,American Restaurant,Movie Theater,Pie Shop,Pharmacy,Pet Supplies Store
6,Didsbury West,53.423368,-2.246432,4,Pub,Bar,Deli,Café,Persian Restaurant,Bakery,Beer Store,Cupcake Shop,Japanese Restaurant,Grocery Store
7,Old Moat,53.435577,-2.235874,2,Coffee Shop,Sushi Restaurant,Park,Café,Grocery Store,Vegan and Vegetarian Restaurant,Turkish Restaurant,Pie Shop,Pharmacy,Pet Supplies Store
8,Burnage,53.426227,-2.203678,0,Supermarket,Convenience Store,Liquor Store,Movie Theater,Pie Shop,Pharmacy,Pet Supplies Store,Persian Restaurant,Performing Arts Venue,Pedestrian Plaza
9,Levenshulme,53.441961,-2.185025,4,Pub,Bar,Antique Store,Indian Restaurant,Pakistani Restaurant,Betting Shop,Grocery Store,Tram Station,Mobile Phone Store,Pharmacy


### Visualising the Manchester's neighbourhood cluster

In [72]:
# Visualizing the resulting clusters
# create map
map_clusters = folium.Map(location=[latitude_man, longitude_man], zoom_start=11)

# set color scheme for the clusters
x = np.arange(kclusters)
ys = [i + x + (i*x)**2 for i in range(kclusters)]
colors_array = cm.rainbow(np.linspace(0, 1, len(ys)))
rainbow = [colors.rgb2hex(i) for i in colors_array]

# add markers to the map
markers_colors = []
for lat, lon, poi, cluster in zip(manchester_merged['Latitude'], manchester_merged['Longitude'], manchester_merged['Neighbourhood'], manchester_merged['Cluster Labels']):
    label = folium.Popup(str(poi) + ' Cluster ' + str(cluster), parse_html=True)
    folium.CircleMarker(
        [lat, lon],
        radius=5,
        popup=label,
        color=rainbow[cluster-1],
        fill=True,
        fill_color=rainbow[cluster-1],
        fill_opacity=0.7).add_to(map_clusters)
       
map_clusters

### Examine clusters

#### Cluster 1

In [73]:
manchester_merged.loc[manchester_merged['Cluster Labels'] == 0, manchester_merged.columns[[0] + list(range(3, manchester_merged.shape[1]))]]

,Neighbourhood,Cluster Labels,1st Most Common Venue,2nd Most Common Venue,3rd Most Common Venue,4th Most Common Venue,5th Most Common Venue,6th Most Common Venue,7th Most Common Venue,8th Most Common Venue,9th Most Common Venue,10th Most Common Venue
2,Moss Side,0,Discount Store,Supermarket,Fuel Station,Farm,Fast Food Restaurant,Pastry Shop,Music Venue,Pakistani Restaurant,Park,Pedestrian Plaza
8,Burnage,0,Supermarket,Convenience Store,Liquor Store,Movie Theater,Pie Shop,Pharmacy,Pet Supplies Store,Persian Restaurant,Performing Arts Venue,Pedestrian Plaza
13,Longsight,0,Supermarket,Turkish Restaurant,Bus Stop,Market,Fast Food Restaurant,Grocery Store,Asian Restaurant,Italian Restaurant,Pie Shop,Pharmacy
17,Higher Blackley,0,Pub,Arts and Entertainment,Fast Food Restaurant,Clothing Store,Hotel,Vietnamese Restaurant,Vegan and Vegetarian Restaurant,Pie Shop,Pharmacy,Pet Supplies Store
25,Baguley,0,Supermarket,Fuel Station,Clothing Store,Hardware Store,Coffee Shop,Pet Supplies Store,Garden,Tram Station,Furniture and Home Store,Donut Shop
27,Harpurhey,0,Fast Food Restaurant,Hotel,Flea Market,Shopping Mall,Eyecare Store,Supermarket,Japanese Restaurant,Museum,Pharmacy,Pet Supplies Store


##### *Cluster 1 seems to be popular for Chinese restaurants, pubs, and convenience store*

#### Cluster 2

In [74]:
manchester_merged.loc[manchester_merged['Cluster Labels'] == 1, manchester_merged.columns[[0] + list(range(3, manchester_merged.shape[1]))]]

,Neighbourhood,Cluster Labels,1st Most Common Venue,2nd Most Common Venue,3rd Most Common Venue,4th Most Common Venue,5th Most Common Venue,6th Most Common Venue,7th Most Common Venue,8th Most Common Venue,9th Most Common Venue,10th Most Common Venue
23,Fallowfield,1,Coffee Shop,Ice Cream Parlor,Indian Restaurant,Lake,Scenic Lookout,Music Venue,Pier,Pie Shop,Pharmacy,Pet Supplies Store
31,Charlestown,1,Coffee Shop,Grocery Store,Pizzeria,Pier,Pie Shop,Pharmacy,Pet Supplies Store,Persian Restaurant,Performing Arts Venue,Pedestrian Plaza


##### *Cluster 2 seems to be popular for different varieties  restaurants, grocery stores and bars/pubs*

### Cluster 3

In [75]:
manchester_merged.loc[manchester_merged['Cluster Labels'] == 2, manchester_merged.columns[[0] + list(range(3, manchester_merged.shape[1]))]]

,Neighbourhood,Cluster Labels,1st Most Common Venue,2nd Most Common Venue,3rd Most Common Venue,4th Most Common Venue,5th Most Common Venue,6th Most Common Venue,7th Most Common Venue,8th Most Common Venue,9th Most Common Venue,10th Most Common Venue
0,City Centre,2,Coffee Shop,Hotel,Bar,Café,Pub,Plaza,Asian Restaurant,Italian Restaurant,Clothing Store,Bookstore
1,Ancoats and Clayton,2,Cosmetics Store,Italian Restaurant,Retail,Park,Mobile Phone Store,American Restaurant,Museum,Pie Shop,Pharmacy,Pet Supplies Store
3,Bradford,2,Tram Station,Canal,Thai Restaurant,Music Venue,American Restaurant,Movie Theater,Pie Shop,Pharmacy,Pet Supplies Store,Persian Restaurant
4,Hulme,2,Convenience Store,Café,Park,Tanning Salon,Pharmacy,Performing Arts Venue,Chinese Restaurant,Pedestrian Plaza,Caribbean Restaurant,Fast Food Restaurant
5,Chorlton,2,Tram Station,Flower Store,Jewelry Store,Park,Music Venue,American Restaurant,Movie Theater,Pie Shop,Pharmacy,Pet Supplies Store
7,Old Moat,2,Coffee Shop,Sushi Restaurant,Park,Café,Grocery Store,Vegan and Vegetarian Restaurant,Turkish Restaurant,Pie Shop,Pharmacy,Pet Supplies Store
10,Didsbury East,2,Pub,Café,Hotel,Movie Theater,Coffee Shop,Sandwich Spot,Casino,Shopping Plaza,Bus Station,Bowling Alley
11,Withington,2,Coffee Shop,Vegan and Vegetarian Restaurant,Café,Grocery Store,Pub,Restaurant,Sushi Restaurant,Sandwich Spot,Middle Eastern Restaurant,Vietnamese Restaurant
12,Chorlton Park,2,Grocery Store,Sports Bar,Bar,Park,Caribbean Restaurant,Falafel Restaurant,Fish and Chips Shop,Fuel Station,Pizzeria,Japanese Restaurant
14,Whalley Range,2,Grocery Store,Cultural Center,Sandwich Spot,Bar,Movie Theater,Pie Shop,Pharmacy,Pet Supplies Store,Persian Restaurant,Performing Arts Venue


##### *Cluster 3 returns only one neighbourhood*

### Cluster 4

In [76]:
manchester_merged.loc[manchester_merged['Cluster Labels'] == 3, manchester_merged.columns[[0] + list(range(3, manchester_merged.shape[1]))]]

,Neighbourhood,Cluster Labels,1st Most Common Venue,2nd Most Common Venue,3rd Most Common Venue,4th Most Common Venue,5th Most Common Venue,6th Most Common Venue,7th Most Common Venue,8th Most Common Venue,9th Most Common Venue,10th Most Common Venue
16,Cheetham,3,Café,American Restaurant,Plaza,Pier,Pie Shop,Pharmacy,Pet Supplies Store,Persian Restaurant,Performing Arts Venue,Pedestrian Plaza


### Cluster 5

In [77]:
manchester_merged.loc[manchester_merged['Cluster Labels'] == 4, manchester_merged.columns[[0] + list(range(3, manchester_merged.shape[1]))]]

,Neighbourhood,Cluster Labels,1st Most Common Venue,2nd Most Common Venue,3rd Most Common Venue,4th Most Common Venue,5th Most Common Venue,6th Most Common Venue,7th Most Common Venue,8th Most Common Venue,9th Most Common Venue,10th Most Common Venue
6,Didsbury West,4,Pub,Bar,Deli,Café,Persian Restaurant,Bakery,Beer Store,Cupcake Shop,Japanese Restaurant,Grocery Store
9,Levenshulme,4,Pub,Bar,Antique Store,Indian Restaurant,Pakistani Restaurant,Betting Shop,Grocery Store,Tram Station,Mobile Phone Store,Pharmacy
15,Moston,4,Pub,Chinese Restaurant,Flower Store,Retail,American Restaurant,Museum,Pie Shop,Pharmacy,Pet Supplies Store,Persian Restaurant
29,Woodhouse Park,4,Pub,Tram Station,Betting Shop,American Restaurant,Movie Theater,Pie Shop,Pharmacy,Pet Supplies Store,Persian Restaurant,Performing Arts Venue


##### *Cluster 5 seems to be popular for Chinese and fast food restaurants and supermakets/markets*

## 4. Results and Discussion

### Use Case Scenario

Suppose a client is looking to book Airbnb in Manchester area. The customer preferences for Airbnb accomodation is as follows: 

Room type: private room

Price: £55

Minimum night: 2

Number of reviews: 100

Availability: 340 days

In addition, the customer prefers accomodation in a neighbourhood with a lot of Chinese and fast food restaurants and supermakets/markets.

We will use the clustering models developed to make recommendations of Airbnb listings to this customer.

In [78]:
# Use the Airbnb preferences to create a feature for prediction the appropriate listing cluster
feature = [[0, 1, 0, 55, 2, 100, 340]]
# Normalised the feature
feature = listings_scaler.transform(feature)
feature

array([[-0.10187926,  1.05125897, -0.12582938, -0.25174411, -0.11632769,
         1.21220338,  1.3953277 ]])

In [79]:
# Use the KMeans model for Airbnb model to predict the clusture
listing_cluster = k_means_man.predict(feature)
listing_cluster

array([9], dtype=int32)

Therefore, Airbnb listings in cluster label 4 will be suggested for the customer. However, since the customer also want listing in neighbouhood with Chinese and fast food restaurants and supermarkets/markets, we will restrict our recommendations to listings in neighbourhood cluster 5(Label 4) - Old Moat, Didsbury East, Moston, Cheetham and Brooklands neighbourhoods.
Let's apply these conditions to the Airbnb listings to extract the recommended accomodations

In [80]:
#subsetDataFrame = dfObj[dfObj['Product'].isin(['Mangos', 'Grapes']) ]
cond = man_listings['neighbourhood'].isin(['Higher Blackley', 'Gorton South', 'Gorton North', 'Harpurhey', 'Sharston']) & (man_listings['bnb_cluster']==4)
Recommended_listings = man_listings[cond]
print('The number of listings found for our customer is', Recommended_listings.shape[0])
Recommended_listings

The number of listings found for our customer is 2


,id,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,availability_365,bnb_cluster
2157,39115529,Harpurhey,53.50719,-2.22036,Shared room,52.0,1,3,74,4
2314,40163345,Harpurhey,53.51148,-2.20578,Shared room,12.0,1,1,0,4


### Visualise recommended listings on map

Finally, let's visualise the recommended accomodations on the map

In [81]:
# create map of Manchester neighbourhoods using recommended Airbnb superimposed
map_rec = folium.Map(location=[latitude_man, longitude_man], zoom_start=11)

# add markers to map
for lat, lng, label, ID in zip(Recommended_listings['latitude'], Recommended_listings['longitude'], Recommended_listings['neighbourhood'], Recommended_listings['id']):
    label = folium.Popup('{},{}'.format(ID,label), parse_html=True)
    folium.CircleMarker(
        [lat, lng],
        radius=5,
        popup=label,
        color='blue',
        fill=True,
        fill_color='#3186cc',
        fill_opacity=0.7,
        parse_html=False).add_to(map_rec)  
    
map_rec

## 5. Conclusions

In this project, we have been able to deploy data science powers of dataset wrangling, machine learning models (K-Means clustering) and geolocation APIs to develop a system to recommend Airbnb listings to our clients looking to book accomodation in Manchester for short stays. We were able to satisfy the problem statement set out at the start of the project, which is to build a segregation and clustering models able to recommend suitable Airbnb in Manchester based on data of Airbnb listings and neighbourhood venues. The capability of the models were demonstrated through a use case scenario. Out of the over 3000 Airbnb listings, our system was able to pick about 50 listings suitable for our client. It should be emphasized that this 50 listings can be further narrowed down for the client by a stricter application of the criteria. 